# 002. Dual-Model Architecture

## Simplified Overview

Instead of building one model for all cases, dual-model architectures build two complementary models: one specializes in handling the easy cases (where prediction is straightforward), and the other focuses on the hard cases (where uncertainty is high). This mirrors human decision-making: simple decisions follow simple rules, complex decisions require deeper analysis.

## Technical Overview

**Dual-Model Architecture** partitions the prediction task into two stages:
1. **Gating Model** (or Selector): Classiﬁes instances as "easy" or "hard" based on feature uncertainty, conﬁdence metrics, or explicit data characteristics.
2. **Expert Models**.
   * **Specialist 1 (Simple Model)**: Fast, interpretable, low-latency model for easy instances
   * **Specialist 2 (Simple Model)**: More sophisticated, higher-capacity model for hard instances

This is a **hierarchical decision structure** distinct from mixture-of-experts (which blends predictions) or cascading classiﬁers (which use sequential conﬁdence thresholds).

## Detailed Explanation

### Motivation

Standard monolithic models:
* Trade off performance across diverse instance diffculties
* Allocate computational budget uniformly (ineffcient)
* Struggle to balance interpretability vs. accuracy

Dual-model architecture addresses this via specialization:

```
Instance → Gating Model → [Easy → Simple Model] OR [Hard → Complex Model] →
Prediction
```

### Gating Strategies

1. **Uncertainty-Based**: Use model conﬁdence (entropy, margin, prediction variance)

```
confidence = max(P(y|x))
difficulty = 1 - confidence
gate = "hard" if difficulty > threshold else "easy"
```

2. **Feature-Based**: Explicit features signal diffculty (e.g., CVE with sparse telemetry)

```
n_features_observed = count(non-missing features)
gate = "hard" if n_features_observed < threshold else "easy"
```

3. **Learned Gate**: A separate classiﬁer predicts diffculty

```
gate_logits = gating_model(x)
difficulty = sigmoid(gate_logits)
```

### Model Selection

| Aspect | Simple Model | Complex Model |
| :--- | :---: | ---: |
| Type | Linear regression, decision tree, shallow NN | Random Forest, Gradient Boosting, Deep NN |
| Latency | 1-10 ms | 10-100 ms |
| Training Cost | Low | High |
| Interpretability | High | Lower |
| Generalization | Risk of underfitting | Risk of overfitting |

### Variants

* **Confidence-Cascaded**: Route instances by model 1 confidence; escalate uncertain cases to model 2
* **Mixture of Experts**: Continuous blending via gating weights rather than hard routing
* **Adaptive Ensemble**: Different ensemble weights per instance type

## Python Libraries

| Library | Purpose |
|:--- | :---:|
| `scikit-learn` | Base models (LogisticRegression, RandomForestClassifier) |
| `xgboost` / `lightgbm` | Advanced ensemble models |
| `scipy.stats` | Uncertainty quantification (entropy, KL divergence) |
| `tensorflow` / `pytorch` | Neural network experts |
| `mlflow` | Track routing metrics and A/B test architectures |

## References
* Jacobs, R. A., et al. (1991). "Adaptive Mixtures of Local Experts." _Neural Computation_, **3**(1), 79-87.
* Spanos, D., & Celikok, T. (2017). "Effortless Classification with Gated Ensembles." _Advances in Data Analysis and Classification_, **11**(4), 679-707.
* Kuncheva, L. I. (2014). _Combining Pattern Classifiers: Methods and Algorithms._ Wiley, 2nd ed.

## Python Implementation Examples

### Imporation of Requisite Python Modules

In [3]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy.stats import entropy

### Dataset Generation: Scaling and Test-Train Split

In [4]:
# Generate synthetic dataset
X, y = make_classification(
    n_samples=2000,
    n_features=30,
    n_informative=15,
    n_redundant=10,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Example 1: Dual-Model Architecture with Uncertainty-Based Gating

In [5]:
# Step 1: Train Simple Model (Logistic Regression)
simple_model = LogisticRegression(max_iter=1000, random_state=42)
simple_model.fit(X_train, y_train)

# Step 2: Train Complex Model (Random Forest)
complex_model = RandomForestClassifier(n_estimators=100, random_state=42)
complex_model.fit(X_train, y_train)

# Step 3: Use Simple Model predictions to identify "hard" instances
simple_proba = simple_model.predict_proba(X_test)
# Entropy as confidence metric: 0 = certain, log(2) ≈ 0.69 = uncertain
simple_entropy = entropy(simple_proba.T)
uncertainty_threshold = np.percentile(simple_entropy, 50)  # Route top 50% uncertain to complex

print(f"Uncertainty threshold (50th percentile): {uncertainty_threshold:.4f}")

# Step 4: Routing Logic
is_hard = simple_entropy > uncertainty_threshold
n_easy = (~is_hard).sum()
n_hard = is_hard.sum()
print(f"Easy instances: {n_easy}, Hard instances: {n_hard}")

# Step 5: Generate predictions using dual routing
y_pred_dual = np.zeros(len(y_test))
y_pred_proba_dual = np.zeros((len(y_test), 2))

# Easy instances → Simple Model
y_pred_dual[~is_hard] = simple_model.predict(X_test[~is_hard])
y_pred_proba_dual[~is_hard] = simple_model.predict_proba(X_test[~is_hard])

# Hard instances → Complex Model
y_pred_dual[is_hard] = complex_model.predict(X_test[is_hard])
y_pred_proba_dual[is_hard] = complex_model.predict_proba(X_test[is_hard])

# ============================================================================
# Evaluate Performance
# ============================================================================

# Baseline: Simple model only
simple_acc = accuracy_score(y_test, simple_model.predict(X_test))
simple_auc = roc_auc_score(y_test, simple_proba[:, 1])
print(f"\nSimple Model Only:")
print(f"  Accuracy: {simple_acc:.4f}")
print(f"  ROC-AUC:  {simple_auc:.4f}")

# Baseline: Complex model only
complex_acc = accuracy_score(y_test, complex_model.predict(X_test))
complex_auc = roc_auc_score(y_test, complex_model.predict_proba(X_test)[:, 1])
print(f"\nComplex Model Only:")
print(f"  Accuracy: {complex_acc:.4f}")
print(f"  ROC-AUC:  {complex_auc:.4f}")

# Dual-Model Architecture
dual_acc = accuracy_score(y_test, y_pred_dual)
dual_auc = roc_auc_score(y_test, y_pred_proba_dual[:, 1])
print(f"\nDual-Model (Uncertainty-Based Gating):")
print(f"  Accuracy: {dual_acc:.4f}")
print(f"  ROC-AUC:  {dual_auc:.4f}")

# Measure computational efficiency
# (In practice, track latency per path)
print(f"\nRouting Summary:")
print(f"  Easy path (Simple): {n_easy/len(y_test)*100:.1f}%")
print(f"  Hard path (Complex): {n_hard/len(y_test)*100:.1f}%")


Uncertainty threshold (50th percentile): 0.4454
Easy instances: 200, Hard instances: 200

Simple Model Only:
  Accuracy: 0.7950
  ROC-AUC:  0.8828

Complex Model Only:
  Accuracy: 0.9200
  ROC-AUC:  0.9759

Dual-Model (Uncertainty-Based Gating):
  Accuracy: 0.9000
  ROC-AUC:  0.9410

Routing Summary:
  Easy path (Simple): 50.0%
  Hard path (Complex): 50.0%


### Example 2: Feature-Based Gating

In [10]:
# Simulate sparse feature availability (e.g., missing telemetry)
X_test_sparse = X_test.copy()
# Mark some instances as sparse (missing features)
sparsity_mask0 = np.random.binomial(1, 0.3, len(X_test))
sparsity_mask1 = np.random.binomial(1, 0.3, X_test.shape)
X_test_sparse[sparsity_mask0] *= sparsity_mask1[sparsity_mask0]
n_features_available = np.sum(X_test_sparse != 0, axis=1)
is_hard_feature = n_features_available < np.percentile(n_features_available, 30)

y_pred_feature_gate = np.zeros(len(y_test))
y_pred_proba_feature_gate = np.zeros((len(y_test), 2))

y_pred_feature_gate[~is_hard_feature] = simple_model.predict(X_test[~is_hard_feature])
y_pred_proba_feature_gate[~is_hard_feature] = simple_model.predict_proba(X_test[~is_hard_feature])

y_pred_feature_gate[is_hard_feature] = complex_model.predict(X_test[is_hard_feature])
y_pred_proba_feature_gate[is_hard_feature] = complex_model.predict_proba(X_test[is_hard_feature])

feature_gate_acc = accuracy_score(y_test, y_pred_feature_gate)
feature_gate_auc = roc_auc_score(y_test, y_pred_proba_feature_gate[:, 1])
print(f"\nDual-Model (Feature-Based Gating):")
print(f"  Accuracy: {feature_gate_acc:.4f}")
print(f"  ROC-AUC:  {feature_gate_auc:.4f}")


Dual-Model (Feature-Based Gating):
  Accuracy: 0.7950
  ROC-AUC:  0.8824
